# LangChain Model I/O: Prompt, Chat Model, Output Parser

Model I/O는 LangChain에서 다음 세 경계를 나눈다.

- **Prompt**: 데이터와 지시를 모델에 전달할 입력으로 만든다.
- **Chat Model**: 완성된 입력을 받아 모델을 호출한다.
- **Output Parser**: 모델 응답을 프로그램이 사용할 자료형으로 바꾼다.

세 단계를 분리하면 같은 모델을 사용해도 입력 형식, 대화 역할과 후속 데이터 처리 규칙을 각각 바꿀 수 있다.

Chat Model은 문자열 또는 message 목록을 받아 보통 `AIMessage`를 반환한다. 반환값은 필요한 정보의 범위에 따라 다르게 사용한다.

- `AIMessage.content`: 문자열 또는 text·reasoning·tool call 같은 content block 목록을 담는다.
- `AIMessage.text`: content에서 text block만 문자열로 꺼낸다.
- `AIMessage` 전체: reasoning, tool call, 사용량 또는 대화 이력을 다음 단계에 전달할 때 보존한다.

이번 노트북은 템플릿·예시·파서를 거쳐 `입력 딕셔너리 → Prompt → AIMessage → list/dict/Pydantic 객체` 흐름을 만든다.

<img src="https://d.pr/i/Wy5B5B+" width="1000" alt="LangChain 구성도에서 Model I/O 영역"/>

왼쪽 초록 영역은 Prompts, Language models, Output parsers가 연결되는 Model I/O를 가리킨다.

이번 단원은 이 영역만 다루며, 전체 그림에는 이전 세대 구성과 명칭이 섞일 수 있으므로 최신 API는 아래의 공식 Models, Prompt, structured output 문서와 코드로 확인한다.


## 환경 준비

- [LangChain Models](https://docs.langchain.com/oss/python/langchain/models)
- [Prompt templates](https://reference.langchain.com/python/langchain-core/prompts)
- [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)


In [1]:
# pydantic : 파이썬 데이터 검증과 설정 관리를 위한 라이브러리
# pip install -U langchain langchain-openai pydantic python-dotenv

## 1. Chat Model은 message를 받고 AIMessage를 돌려준다

Chat Model에는 입력 목적에 따라 다음 두 가지 형태를 전달한다.

- **문자열**: 독립적인 질문이나 지시처럼 대화 맥락이 필요하지 않을 때 사용한다.
- **message 목록**: 이전 대화나 역할별 지시를 함께 전달할 때 사용한다.

message 목록의 각 항목에는 텍스트의 역할이 지정된다.

- `system`: 모델이 따라야 할 행동 기준과 답변 방식을 정한다.
- `human` 또는 `user`: 사용자의 질문이나 요청을 나타낸다.
- `ai` 또는 `assistant`: 이전에 모델이 생성한 응답을 나타낸다.

역할은 단순한 라벨이 아니라 대화에서 각 텍스트를 해석하는 위치를 정한다.

`ChatOpenAI`는 `invoke()` 뒤 `AIMessage`를 반환한다. Responses API의 `content`에는 text·reasoning 같은 block이 함께 들어갈 수 있다.

- 답변 문자열만 표시할 때는 `.text`를 사용한다.
- 도구 호출, 사용량 또는 대화 이력을 이어 갈 때는 message 객체 전체를 보존한다.

자세한 구조는 [LangChain Messages](https://docs.langchain.com/oss/python/langchain/messages)에서 확인한다.


In [1]:
import os

from dotenv import load_dotenv
from google.protobuf import descriptor
from langchain_openai import ChatOpenAI

load_dotenv()
OPENAI_CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")



In [3]:
# OpenAI ChatModel인 ChatOpenAI 생성
model = ChatOpenAI(model_name=OPENAI_CHAT_MODEL,
                   use_responses_api=True # Responses API 사용
                   )
# ChatModel에 전달되는 단순 문자열은 HumanMessage로 해석된다.
question= "투르크메니스탄의 수도는 어디야?"

# invoke() : 입력 하나를 처리하고 content, text, usage 등을 담은 AIMessage 객체를 반환한다.
response = model.invoke(question)

print(type(response).__name__)
print(response.content)
print(response.text)

AIMessage
[{'type': 'text', 'text': '투르크메니스탄의 수도는 **아시가바트(Aşgabat)**입니다.', 'annotations': [], 'id': 'msg_050d49ffa223af3b006a7532d08acc819ab6c8fbccca955cd3', 'phase': 'final_answer'}]
투르크메니스탄의 수도는 **아시가바트(Aşgabat)**입니다.


### 역할이 있는 message 목록 만들기

아래 셀은 API를 호출하지 않고 `SystemMessage`, `HumanMessage`, `AIMessage`가 각각 어떤 대화 위치를 차지하는지 확인한다. 실제 호출에서는 이 목록 전체를 `model.invoke(messages)`에 전달해 이전 응답까지 포함한 대화를 이어 갈 수 있다.


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# SystemMessage: 대화 전체의 행동 기준 또는 규칙
# HumanMessage : 사용자 요청
# AIMessage:LLM 응답
# + 다음 HumanMessage가 이를 문맥으로 참조할 수 있게 한다.

messages = [
    SystemMessage('모든 답변은 한 문장으로 작성하고 극존칭을 사용한다.'),
    HumanMessage("LangChain은 무엇인가요?"),
    AIMessage("LLM 애플리케이션의 구성 요소를 연결하는 프레임 워크입니다."),
    HumanMessage('그 중 Model I/O의 역할은 무엇인가요?')
]

for message in messages :
    print(f'{message.type}:{message.content}')

system:모든 답변은 한 문장으로 작성하고 극존칭을 사용한다.
human:LangChain은 무엇인가요?
ai:LLM 애플리케이션의 구성 요소를 연결하는 프레임 워크입니다.
human:그 중 Model I/O의 역할은 무엇인가요?


### invoke, batch, stream은 언제 다른가

세 호출 방식은 입력 개수와 결과를 받는 시점이 다르다.

- `invoke()`: 입력 하나의 생성이 끝날 때까지 기다린 뒤 최종 `AIMessage` 하나를 받는다.
- `batch()`: 독립적인 여러 입력을 병렬로 처리하고 입력 순서에 맞는 `AIMessage` 목록을 받는다. LangChain의 클라이언트 측 병렬화이므로 공급자의 별도 Batch API와 다르다.
- `stream()`: 생성 중인 `AIMessageChunk`를 순서대로 받아 긴 답변의 첫 부분부터 화면에 보여 준다.

공식 Models 문서의 호출 방식과 반환 차이를 기준으로, 요청끼리 문맥을 공유하면 `batch()`가 아니라 각 대화 이력을 따로 구성한다. parser가 완성된 JSON을 요구할 때는 chunk를 모두 합친 다음 파싱해야 한다.


In [5]:
# invoke()
question='LangChain에서 batch()가 무엇인지 두 문장과 간편한 예시로 설명'
single=model.invoke(question)
print("invoke",single.text)

invoke LangChain의 `batch()`는 하나의 Runnable(체인·모델 등)에 여러 입력을 한 번에 전달해 결과 목록을 받는 메서드입니다.  
예: `chain.batch(["안녕", "날씨 어때?"])`처럼 호출하면 각 입력에 대한 결과가 리스트로 반환됩니다.


In [6]:
single=model.invoke(messages)
print(single.text)

LangChain의 Model I/O는 프롬프트를 구성하고 언어 모델을 호출하며 모델의 출력을 필요한 형식으로 변환·파싱하는 기능을 담당합니다.


In [7]:
# batch() : 질문 목록을 한 번에 전달하여 병렬로 처리
question = [
    "langChain Prompt의 역할을 한 문장으로 설명",
    'LangChain ChatModel의 역할을 한 문장으로 설명',
    'LangChain Output Parser의 역할을 한 문장으로 설명'
]


# 'max_concurrency' : 동시에 병렬로 처리할 요청(질문) 수
batched = model.batch(
    question,
    config = {'max_concurrency':3}
)

for item in batched:
    print('batch',type(item).__name__,item.text)



batch AIMessage LangChain의 Prompt는 LLM에 전달할 질문과 지침을 일관된 형식으로 구성하는 역할을 합니다.
batch AIMessage LangChain의 ChatModel은 사용자 입력을 바탕으로 대화형 언어 모델의 응답을 생성하고 관리하는 역할을 합니다.
batch AIMessage LangChain Output Parser는 LLM의 출력을 애플리케이션에서 사용하기 쉬운 구조화된 형식으로 변환하고 검증하는 역할을 합니다.


In [8]:
# stream() : AIMessageChunk를 순서대로 내보냄
full = None
question = 'LangChain ChatModel.stream()의 장점을 짧게 설명'
for chunk in model.stream(question):
    full = chunk if full is None else full + chunk

    print(chunk.text, end='',flush=True)

print(type(full).__name__)

LangChain의 `ChatModel.stream()`은 모델의 응답을 한 번에 기다리지 않고 **생성되는 토큰(청크)을 실시간으로 전달**합니다.

장점:
- 첫 응답이 빠르게 보여 사용자 체감 대기 시간 감소
- 긴 답변도 생성 과정을 실시간 표시 가능
- 진행 상태나 중간 결과를 UI에 즉시 반영
- 스트리밍 기반 챗봇 구현에 적합AIMessageChunk


### Hugging Face와 ModelLaboratory의 위치

LangChain은 여러 모델 공급자를 같은 Runnable 흐름으로 연결할 수 있다. 공급자를 바꿔도 Prompt·Model·Parser의 경계는 그대로이다.

- `HuggingFacePipeline.from_model_id()`: 로컬 모델 다운로드와 실행 하드웨어가 필요하다.
- `HuggingFaceEndpoint`: 지원 모델, 인증 토큰, 이용 요금을 확인해야 한다.
- 최신 공급자 통합: [LangChain integrations](https://docs.langchain.com/oss/python/integrations/chat)에서 확인한다.

Model I/O 첫 학습에서는 무거운 모델을 실행하지 않고 공통 인터페이스와 선택 기준에 집중한다.

`ModelLaboratory`는 과거 여러 LLM의 응답을 비교할 때 사용하던 utility이다. 현재는 각 provider integration을 같은 입력으로 직접 호출해 비교한다. 이 방식은 결과·비용·지연 시간을 어떤 기준으로 비교하는지 코드에 명확히 남길 수 있다.


## 2. Prompt는 데이터에서 모델 입력을 만든다

두 Prompt template는 같은 변수 딕셔너리를 서로 다른 모델 입력으로 바꾼다.

- `PromptTemplate`: 변수 딕셔너리를 문자열 하나로 바꾼다. 한 문장 생성처럼 역할 구분이 필요 없는 입력에 적합하다.
- `ChatPromptTemplate`: 변수 딕셔너리를 역할이 있는 message 목록으로 바꾼다. system 지시, 사용자 질문과 이전 대화를 구분할 때 사용한다.

모델을 호출하기 전에 template의 `invoke()` 결과를 출력하면 변수 이름, 역할, 문장 순서를 무료로 검토할 수 있다. 이 확인은 실제 모델 호출 전에 입력 오류를 찾는 가장 짧은 방법이다.


### PromptTemplate의 변수 교체

`product` 값만 바꾸어도 동일한 문장 틀을 재사용할 수 있다. 출력은 `StringPromptValue`이며, 여기의 `.text`가 뒤에서 Chat Model에 들어갈 실제 문자열이다.


In [9]:
from langchain_core.prompts import PromptTemplate, few_shot

ad_prompt = PromptTemplate.from_template(
    '{product}를 소개하는 광고 문구를 세줄로 작성한다.'
)

camera_prompt =ad_prompt.invoke({'product':'카메라'})

fridge_prompt = ad_prompt.invoke({'product':'삼성 비스포크'})

print(camera_prompt.text)
print(fridge_prompt.text)

카메라를 소개하는 광고 문구를 세줄로 작성한다.
삼성 비스포크를 소개하는 광고 문구를 세줄로 작성한다.


### ChatPromptTemplate는 역할과 변수를 함께 관리한다

ChatPromptTemplate는 `domain`, `question` 같은 입력을 받아 `SystemMessage`, `HumanMessage` 순서의 `ChatPromptValue`를 만든다. 문자열 template에 system 지시를 섞어 쓰는 것보다, 어떤 문장이 행동 기준이고 어떤 문장이 사용자 질문인지 명확하다.


In [10]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ('system','당신은 {domain}분야의 전문가입니다.'),
    ('human','{question}')
])

prompt_value = chat_prompt.invoke({
    'domain':'AI Engineer',
    'question':'AI 엔지니어가 하는 일이 무엇인가'
})

for message in prompt_value.messages:
    print(f"{message.type} : {message.content}")

system : 당신은 AI Engineer분야의 전문가입니다.
human : AI 엔지니어가 하는 일이 무엇인가


### Few-shot은 답의 내용보다 답의 형식을 보여 준다

Few-shot prompt는 모델에게 몇 개의 입력·출력 예시를 먼저 보여 준 뒤 새 입력을 준다. 목적은 모델의 일반 지식을 저장하는 것이 아니라, 이번 요청에서 따를 분류 라벨·답변 형식·추론 패턴을 구체적으로 보여 주는 데 있다.

좋은 예시는 다음 조건을 만족한다.

- 실제 입력과 같은 형식을 사용한다.
- 서로 다른 상황을 대표한다.
- 새 질문의 정답을 그대로 누설하지 않는다.

예시가 많아질수록 토큰 비용과 서로 충돌할 위험도 커지므로 최소 대표 예시부터 시작한다.


In [11]:
from langchain_core.prompts import FewShotPromptTemplate
# 모델에게 보여 줄 입력·출력 사례이며 key는 example_prompt의 q·a 변수와 일치해야 한다.
examples = [{"q": "2 + 2 = ?", "a": "4"}, {"q": "3 + 5 = ?", "a": "8"}]

example_prompt = PromptTemplate.from_template(
    "Q:{q}\nA:{a}",
)

few_shot_prompt = FewShotPromptTemplate(
    examples = examples, #예시
    example_prompt = example_prompt, # 예시가 사용되는 프롬프트
    prefix = "다음 수학 문제는 정답만 출력한다.", # 접두사
    suffix = "Q:{question}\nA", # 접미사
    input_variables=['question'] # 실행 시 받아야할 kay값
)


# invoke() 수행
# prefix -> 변환된 examples -> suffix 순서로 하나의 StringPromptValue가 새성된다.
formatted_few_shot = few_shot_prompt.invoke({'question':'123+456=?'})
print(formatted_few_shot.text)

다음 수학 문제는 정답만 출력한다.

Q:2 + 2 = ?
A:4

Q:3 + 5 = ?
A:8

Q:123+456=?
A


## 3. Output Parser는 생성 텍스트를 후속 코드의 자료형으로 바꾼다

모델이 만든 텍스트는 사람이 읽기에는 충분해도 반복문, JSON 저장, API 응답에는 불편할 수 있다. Output Parser는 문자열 또는 `AIMessage`의 text 출력을 `list`, `dict` 같은 Python 객체로 변환해 다음 코드의 입력 계약을 만든다. parser는 형식을 검증할 뿐, 모델이 말한 사실이 참인지 검증하지는 않는다.

Parser는 필요한 Python 자료형에 따라 선택한다.

- `CommaSeparatedListOutputParser`: 쉼표로 구분된 텍스트를 Python 목록으로 바꾼다.
- `JsonOutputParser`: JSON 객체나 배열을 Python dict/list로 바꾼다.

두 parser 모두 모델에 출력 형식을 지시하는 단계와 반환값을 파싱하는 단계를 함께 설계해야 한다.


### parser에 문자열을 직접 넣기

먼저 API 호출 없이 parser 자체를 검증한다. 이 순서라면 오류가 생겼을 때 모델 생성 문제인지, 형식 지시 문제인지, parser 규칙 문제인지 분리할 수 있다.


In [12]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

#문자열 내 쉼표 (,)를 기준으로 리스트 요소를 나누고, 앞뒤 공백을 정리함 ㅇ
list_parser = CommaSeparatedListOutputParser()

fruit_list = list_parser.parse('사과, 오렌지, 포도, 바나나, 키위, 두리안')
print(fruit_list)

['사과', '오렌지', '포도', '바나나', '키위', '두리안']


### JsonOutputParser로 JSON 계약을 확인하기

JSON 문자열은 사람이 읽기에는 명확해 보여도 Python에서는 아직 문자열이다. `JsonOutputParser`는 JSON 문법을 읽어 dict 또는 list로 바꾸며, 다음 코드가 `book["tags"]`처럼 key에 접근할 수 있게 한다.


In [13]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

# json 파싱 진행 시
# 전체 객체-> dict
# 내부 {},[] -> dict, list 변환
book = json_parser.parse('{"title": "LLM 입문", "pages": 250, "tags": ["AI", "LangChain"]}')
print(book)

print(type(book).__name__, type(book["tags"]).__name__)

{'title': 'LLM 입문', 'pages': 250, 'tags': ['AI', 'LangChain']}
dict list


### Prompt → Model → Parser를 LCEL로 연결하기

LCEL(LangChain Expression Language)의 `|` 연산자는 앞 단계의 출력을 다음 단계의 입력으로 연결한다. 아래 chain의 흐름은 `{"subject": ...} → StringPromptValue → AIMessage → list[str]`이다.

- **단계별 실행**: Prompt, Model과 Parser를 따로 호출하므로 오류가 발생한 위치를 확인하기 쉽다.
- **LCEL chain**: 같은 처리 흐름을 하나의 Runnable로 묶어 반복 실행하거나 다른 chain과 연결하기 쉽다.


In [14]:
# partial_variables : 미리 고정할 값을 대입
# input_variables : invoke시 전달될 값을 지정
# get_format_instructions() : 쉼표 구분 규칙


team_prompt = PromptTemplate(
    template='{subject}팀{n}개를 제시한다.\n {format_instructions}',
    input_variables = ['subject', 'n'],
    partial_variables={
        'format_instructions':list_parser.get_format_instructions()
    }


)

team_chain = team_prompt | model | list_parser

teams = team_chain.invoke({'subject':"g한국 프로 야구", "n":5})

print(teams)

['LG 트윈스', '두산 베어스', '삼성 라이온즈', 'KIA 타이거즈', '롯데 자이언츠']


## 4. Parser 방식과 provider-native structured output의 선택 경계

출력 구조가 단순한지, 필드 계약이 필요한지에 따라 방법을 선택한다.

- 문자열 parser: 쉼표 목록이나 이미 생성된 JSON 텍스트를 Python 자료형으로 바꿀 때 적합하다.
- provider-native structured output: 필드 이름과 타입이 반드시 맞아야 할 때 우선한다.

공식 LangChain 문서는 공급자가 schema를 직접 강제할 수 있을 때 더 신뢰할 수 있는 방법이라고 설명한다.

provider-native structured output에도 한계가 있다. 모델·provider의 지원 여부를 확인하고 schema를 올바르게 설계해야 한다. 형식은 보장해도 내용의 사실성까지 보장하지는 않는다.

provider-native schema를 지원하지 않으면 해당 chat-model 통합의 `function_calling` 등 structured-output 방식을 확인한다. 아래 코드는 `Prompt → Chat Model → Pydantic schema`를 결합하며, 실행 시 유료 API를 호출한다.


### Prompt → Model → structured output 연결하기

레시피 요청을 역할이 있는 prompt로 만들고, Pydantic `Recipe` schema를 `with_structured_output()`에 전달한다. model이 지원하는 provider-native structured output을 이용할 수 있으면 schema를 따르는 객체를 받고, 그렇지 않은 공급자는 그 통합의 structured-output 지원 방식을 확인해야 한다.


In [15]:
from pydantic import BaseModel, Field

# BaseModel을 상속하면 필드 이름·자료형·설명이 모델의 출력 schema가 된다.
class Recipe(BaseModel):

    # title은 문자열 하나이고 ingredients와 steps는 문자열 여러 개를 담는 list[str].
    # Field의 description은 모델이 각 필드에 어떤 값을 채울지 판단하는 설명으로 사용됨
    title: str = Field(description="레시피 이름")
    ingredients: list[str] = Field(description="필요한 재료 목록")
    steps: list[str] = Field(description="조리 순서")

# human의 {ingredient}에는 invoke() 딕셔너리 값이 들어가고 system은 모든 요청에 공통으로 적용된다.
recipe_prompt = ChatPromptTemplate.from_messages([
    ("system", "재료와 조리 순서를 간결하게 정리하는 요리 도우미이다."),
    ("human", "{ingredient}로 만들 수 있는 간단한 요리를 추천한다."),
])

# with_structured_output(Recipe)는 모델 출력을 Recipe schema에 맞춰 검증하고 Recipe 객체로 반환한다.
schema_model = model.with_structured_output(Recipe)

# Prompt의 ChatPromptValue가 schema_model로 전달되므로 chain의 최종 결과는 AIMessage가 아니라 Recipe이다.
recipe_chain = recipe_prompt | schema_model

# 입력 dict → ChatPromptValue → Recipe 객체 순서로 변환하며 이 호출에서 네트워크와 유료 API를 사용한다.
recipe = recipe_chain.invoke({"ingredient": "토마토와 달걀"})
# model_dump()는 Recipe를 JSON 저장이나 DataFrame 생성에 사용할 수 있는 Python dict로 변환한다.
print(recipe.model_dump())

{'title': '토마토 달걀볶음', 'ingredients': ['토마토 2개', '달걀 3개', '대파 약간', '식용유 1큰술', '소금 약간', '후추 약간', '설탕 1/2작은술(선택)'], 'steps': ['토마토는 먹기 좋은 크기로 썰고, 달걀은 소금과 함께 풀어 둔다.', '팬에 식용유를 두르고 달걀을 넣어 반숙으로 볶은 뒤 잠시 덜어 둔다.', '같은 팬에 대파와 토마토를 넣고 2~3분 볶는다.', '토마토가 살짝 부드러워지면 달걀을 다시 넣고 가볍게 섞는다.', '소금, 후추로 간하고 원하면 설탕을 조금 넣어 완성한다.']}


## 고객 문의 자동 분류·답변 초안 생성기

앞에서는 하나의 입력을 parser나 Pydantic 객체로 변환했다. 이제 고객 문의 여러 건으로 Model I/O의 전체 흐름을 확인한다.

- **Prompt**: 문의 내용과 분류 규칙을 모델 입력으로 만든다.
- **Chat Model과 `TicketAnalysis`**: 문의를 처리하고 정해진 schema의 객체를 반환한다.
- **Python 후처리**: 객체를 DataFrame으로 바꾸고 기준값과 비교한다.

아직 Retrieval을 사용하지 않으므로 분류 규칙은 system message에 직접 제공한다. `batch()`는 여러 입력을 편리하게 처리하지만 공급자의 단일 Batch API가 아니라 독립적인 모델 요청을 병렬로 실행하므로 문의 건수만큼 API 사용량이 발생한다.


### 문의 데이터와 비교 기준 준비하기

각 문의에는 용도가 다른 두 종류의 값이 있다.

- **모델 입력**: `ticket_id`, `text`
- **결과 비교 기준**: `expected_category`, `expected_urgency`

기대값은 Prompt에 넣지 않고 모델 출력과 나중에 비교한다. 이 값은 분류 규칙이 적용되는지 확인하는 소규모 기준이며 일반화 성능을 추정하기 위한 홀드아웃 데이터와는 목적이 다르다.


In [16]:
from typing import Literal

import pandas as pd

# expected 값은 모델에게 전달하지 않고 결과 비교에만 사용한다.
ticket_cases = [
    {
        "ticket_id": "T001",
        "text": "결제가 두 번 됐는데 하나는 취소해 주세요.",
        "expected_category": "결제",
        "expected_urgency": "높음",
    },
    {
        "ticket_id": "T002",
        "text": "배송 완료라고 나오는데 상품을 받지 못했어요.",
        "expected_category": "배송",
        "expected_urgency": "높음",
    },
    {
        "ticket_id": "T003",
        "text": "비밀번호를 잊어버려서 로그인할 수 없습니다.",
        "expected_category": "계정",
        "expected_urgency": "보통",
    },
    {
        "ticket_id": "T004",
        "text": "제품 색상이 생각했던 것과 달라서 반품하고 싶어요.",
        "expected_category": "반품",
        "expected_urgency": "보통",
    },
    {
        "ticket_id": "T005",
        "text": "선물 포장도 가능한가요?",
        "expected_category": "기타",
        "expected_urgency": "낮음",
    },
]

case_frame = pd.DataFrame(ticket_cases)
print(case_frame.to_string(index=False))


ticket_id                         text expected_category expected_urgency
     T001     결제가 두 번 됐는데 하나는 취소해 주세요.                결제               높음
     T002    배송 완료라고 나오는데 상품을 받지 못했어요.                배송               높음
     T003     비밀번호를 잊어버려서 로그인할 수 없습니다.                계정               보통
     T004 제품 색상이 생각했던 것과 달라서 반품하고 싶어요.                반품               보통
     T005                선물 포장도 가능한가요?                기타               낮음


### 반환 계약과 분류 기준 정의하기

`TicketAnalysis`는 모델이 반드시 채워야 할 필드와 허용 값을 정한다. 주요 필드의 역할은 다음과 같다.

- `category`, `urgency`: `Literal`로 허용된 문자열만 반환하게 제한한다.
- `summary`: 문의의 핵심을 한 문장으로 정리한다.
- `required_information`: 처리에 필요하지만 문의 원문에 없는 정보를 기록한다.
- `reply_draft`: 처리 완료를 단정하지 않는 고객용 답변 초안을 만든다.

schema는 필드의 존재와 자료형을 검증하지만 분류가 실제 기준과 맞는지는 보장하지 않는다. 따라서 다음 단계에서 기대값과 별도로 비교한다.


In [17]:
from pydantic import BaseModel, Field
from typing import Literal #허용 문자열 제한
from langchain_core.prompts import ChatPromptTemplate



model = ChatOpenAI(model_name=OPENAI_CHAT_MODEL,
                   use_responses_api=True # Responses API 사용
                   )

#LLM이 입력을 받았을 떄 응답으로 출력할 양식
class TicketAnalysis(BaseModel) :
    ticket_id : str = Field(description='입력으로 받은 문의 ID')

    category : Literal['결제','배송','계정','반품','기타'] = Field(description='문의의 대표 범주')

    urgency : Literal['낮음','보통','높음'] = Field(description='업무 처리 우선순위')

    summary : str = Field(description='문의 핵심을 한 문장으로 요약한 내용')

    required_information : list[str] = Field(description ='처리에 필요하지만 문의에 없는 정보, 없으면 빈 목록')

    reply_draft : str = Field(description='처리 완료를 단정하지 않는 두 문장 이내의 답변 초안 ')

#입력용 프롬프트
ticket_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """고객 문의를 분류하고 상담 답변 초안을 작성한다.
분류 기준:
- 중복 결제나 승인 오류는 결제·높음으로 분류한다.
- 배송 완료 후 미수령은 배송·높음으로 분류한다.
- 로그인이나 비밀번호 문제는 계정·보통으로 분류한다.
- 단순 변심이나 색상 차이 반품은 반품·보통으로 분류한다.
- 정보 문의는 기타·낮음으로 분류한다.
입력에 없는 주문번호·결제번호·처리 결과를 지어내지 않는다.
입력 ticket_id를 그대로 반환하고 답변 초안은 두 문장 이내로 작성한다.""",
    ),
    ("human", "ticket_id: {ticket_id}\n고객 문의: {text}"),
])

#모델(LLM)의 생성 결과를 TicketAnalysis 필드와 자료형에 맞춰서 반환
ticket_schema_model = model.with_structured_output(TicketAnalysis)

#chain에 입력(dict) -> ChatPromptTemplate(Message)
# -> ChatModel(LLM) -> 출력(TicketAnalysis)
ticket_chain = ticket_prompt | ticket_schema_model





### 여러 문의를 batch로 처리하기

`batch()`를 실행할 때 확인할 값은 다음과 같다.

- **입력**: 모델에는 `ticket_id`와 `text`만 전달하고 기대값은 제외한다.
- **출력**: 입력 목록과 같은 순서의 `TicketAnalysis` 목록을 반환한다.
- **`max_concurrency=3`**: 동시에 처리할 최대 요청 수이며 결과 개수나 API 호출 횟수를 줄이지 않는다.


In [18]:
#입력용 문의 5건 준비
ticket_inputs = [
    {"ticket_id": case["ticket_id"], "text": case["text"]}
    for case in ticket_cases
]

#하나의 대답을 원하면invoke
# 문의가 여러개니까 batch
# batch 입력은 각 입력이 독립적으로 API를 요쳥함

ticket_results = ticket_chain.batch(
    ticket_inputs,
    max_concurrency=3, #3개씩 병렬 처리
)

for result in ticket_results:
    print("ticket_id:", result.ticket_id)
    print("category:", result.category)
    print("urgency:", result.urgency)
    print("summary:", result.summary)
    print("="*100)





ticket_id: T001
category: 결제
urgency: 높음
summary: 동일 결제가 중복 처리되어 한 건의 취소를 요청함
ticket_id: T002
category: 배송
urgency: 높음
summary: 배송 완료로 표시되지만 상품을 수령하지 못함
ticket_id: T003
category: 계정
urgency: 보통
summary: 비밀번호를 잊어버려 로그인할 수 없음
ticket_id: T004
category: 반품
urgency: 보통
summary: 제품 색상이 기대와 달라 반품을 요청함
ticket_id: T005
category: 기타
urgency: 낮음
summary: 선물 포장 가능 여부를 문의함


### 구조화된 결과를 DataFrame으로 검증하기

Pydantic 객체를 `model_dump()`로 딕셔너리로 바꾸면 DataFrame 행으로 사용할 수 있다. category와 urgency의 기대값을 ticket_id로 연결한 뒤 일치 여부를 계산한다. schema 통과 여부와 분류 규칙의 일치 여부는 다른 문제이므로 둘을 함께 확인해야 한다. 여기서 계산하는 값은 5개 사례의 규칙 일치율이며 모델의 일반화 성능을 뜻하지 않는다.


In [19]:
ticket_frame = pd.DataFrame([result.model_dump() for result in ticket_results])

expected_category = {case["ticket_id"]: case["expected_category"] for case in ticket_cases}
expected_urgency = {case["ticket_id"]: case["expected_urgency"] for case in ticket_cases}
ticket_frame["expected_category"] = ticket_frame["ticket_id"].map(expected_category)
ticket_frame["expected_urgency"] = ticket_frame["ticket_id"].map(expected_urgency)

# 예측값과 기대값을 행별로 비교해 일치하면 True, 다르면 False인 Booleaan 열을 만든다.
ticket_frame["category_match"] = (
    ticket_frame["category"] == ticket_frame["expected_category"]
)
ticket_frame["urgency_match"] = (
    ticket_frame["urgency"] == ticket_frame["expected_urgency"]
)

comparison_columns = [
    "ticket_id",
    "category",
    "expected_category",
    "category_match",
    "urgency",
    "expected_urgency",
    "urgency_match",
]
display(ticket_frame[comparison_columns])

# 각 Boolean 열의 mean()을 계산해 category와 urgency의 규칙 일치율을 따로 출력
print({
    "category_match_rate": ticket_frame["category_match"].mean(),
    "urgency_match_rate": ticket_frame["urgency_match"].mean(),
})

,ticket_id,category,expected_category,category_match,urgency,expected_urgency,urgency_match
0,T001,결제,결제,True,높음,높음,True
1,T002,배송,배송,True,높음,높음,True
2,T003,계정,계정,True,보통,보통,True
3,T004,반품,반품,True,보통,보통,True
4,T005,기타,기타,True,낮음,낮음,True


{'category_match_rate': np.float64(1.0), 'urgency_match_rate': np.float64(1.0)}


In [22]:
for result in ticket_results:
    print("ticket_id:", result.ticket_id)
    print("category:", result.category)
    print("urgency:", result.urgency)
    print("summary:", result.summary)
    print("reply_draft:", result.reply_draft)
    print("="*100)


ticket_id: T001
category: 결제
urgency: 높음
summary: 동일 결제가 중복 처리되어 한 건의 취소를 요청함
reply_draft: 중복 결제 여부를 확인한 후 한 건의 취소를 진행하겠습니다. 확인을 위해 주문번호 또는 결제번호를 알려 주세요.
ticket_id: T002
category: 배송
urgency: 높음
summary: 배송 완료로 표시되지만 상품을 수령하지 못함
reply_draft: 배송 완료로 표시되지만 상품을 받지 못하셨군요. 주문번호와 배송지 정보를 확인해 주시면 배송 상황을 확인해 드리겠습니다.
ticket_id: T003
category: 계정
urgency: 보통
summary: 비밀번호를 잊어버려 로그인할 수 없음
reply_draft: 로그인 화면의 ‘비밀번호 찾기’ 기능을 통해 비밀번호를 재설정해 주세요. 재설정이 어려우시면 추가로 문의해 주시면 안내해 드리겠습니다.
ticket_id: T004
category: 반품
urgency: 보통
summary: 제품 색상이 기대와 달라 반품을 요청함
reply_draft: 색상이 기대와 다르셨군요. 반품 가능 여부와 절차를 안내해 드리겠습니다.
ticket_id: T005
category: 기타
urgency: 낮음
summary: 선물 포장 가능 여부를 문의함
reply_draft: 선물 포장 서비스 제공 여부를 확인해 안내해 드리겠습니다.


## 회의록 업무 정리기

고객 문의 처리와 같은 Model I/O 구조를 회의록에 적용한다. 회의록 원문에서 요약, 결정 사항과 후속 업무를 추출하되 원문에 없는 담당자나 기한을 추측하지 않는다.

완성할 처리 흐름은 다음과 같다.

1. `MeetingAction`에 업무 내용, 담당자와 기한을 정의한다.
2. `MeetingAnalysis`에 meeting_id, 요약, 결정 사항, 후속 업무와 누락 정보를 정의한다.
3. 회의록만 근거로 사용하도록 `ChatPromptTemplate`[을 작성한다.]()
4. `with_structured_output()`과 `batch()`로 두 회의록을 처리한다.
5. 중첩된 action item을 행 단위 DataFrame으로 평탄화한다.
6. `None`과 대표적인 누락 표현을 같은 값으로 정규화하고 추가 확인 대상으로 분리한다.


### 회의록 입력 준비하기

두 회의록은 누락 정보의 유무가 다르다.

- **첫 번째 회의록**: 모든 후속 업무에 담당자와 기한이 포함되어 있다.
- **두 번째 회의록**: 일부 업무의 담당자나 기한이 빠져 있다.

모델이 빠진 값을 지어내지 않고 `None`과 누락 정보로 표현하는지 확인한다.


In [3]:
# M001은 모든 담당자·기한이 있고, M002는 기한이 없는 업무와 담당자가 없는 업무를 포함한다.
# 두 사례의 차이는 뒤에서 null 처리와 누락 정보 추출을 확인하는 기준이 된다.


meeting_cases = [
    {
        "meeting_id": "M001",
        "notes": (
            "결제 페이지 오류를 이번 배포에서 수정하기로 했다. "
            "민지는 8월 12일까지 오류 화면을 다시 설계하고, "
            "준호는 8월 14일까지 결제 API 재시도 로직을 수정한다."
        ),
    },
    {
        "meeting_id": "M002",
        "notes": (
            "추천 기능은 이번 배포에서 제외하기로 했다. "
            "소연이 사용자 설문을 준비하지만 마감일은 정하지 않았다. "
            "API 비용을 검토해야 하지만 담당자는 아직 정하지 않았다."
        ),
    },
]

# 딕셔너리 key로 회의 ID와 원문을 꺼내 모델에 전달될 두 입력을 실행 전에 확인한다.
for meeting in meeting_cases:
    print(meeting["meeting_id"], meeting["notes"])

M001 결제 페이지 오류를 이번 배포에서 수정하기로 했다. 민지는 8월 12일까지 오류 화면을 다시 설계하고, 준호는 8월 14일까지 결제 API 재시도 로직을 수정한다.
M002 추천 기능은 이번 배포에서 제외하기로 했다. 소연이 사용자 설문을 준비하지만 마감일은 정하지 않았다. API 비용을 검토해야 하지만 담당자는 아직 정하지 않았다.


In [5]:
from pydantic import BaseModel, Field
from typing import Literal #허용 문자열 제한
from langchain_core.prompts import ChatPromptTemplate

# MeetingAction은 회의에서 추출한 후속 업무 한 건의 중첩 schema이다.
# owner·due_date의 str | None은 문자열 또는 null을 허용한다는 뜻이다.
class MeetingAction(BaseModel):
    task: str = Field(description="회의 이후 수행할 구체적인 업무")
    owner: str | None = Field(description="원문에 적힌 담당자, 없으면 null")
    due_date: str | None = Field(description="원문에 적힌 기한, 없으면 null")

# MeetingAnalysis는 회의 한 건의 요약·결정 사항과 MeetingAction 목록을 함께 반환한다.
class MeetingAnalysis(BaseModel):
    meeting_id: str = Field(description="입력으로 받은 회의 ID")
    summary: str = Field(description="회의 핵심을 한 문장으로 요약한 내용")
    decisions: list[str] = Field(description="회의에서 확정한 결정 사항")
    action_items: list[MeetingAction] = Field(description="회의 이후 수행할 업무 목록")
    missing_information: list[str] = Field(
        description="담당자나 기한처럼 원문에서 확인할 수 없는 정보"
    )

# system은 결정 사항과 후속 업무의 구분 기준을 제공하고 원문에 없는 값의 추측을 금지한다.
# human의 meeting_id·notes에는 meeting_cases의 각 딕셔너리 값이 대입된다.
meeting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """회의록만 근거로 결정 사항과 후속 업무를 정리한다.
확정된 방향은 decisions에, 사람이 수행할 일은 action_items에 기록한다.
담당자나 기한이 원문에 없으면 추측하지 말고 null로 반환하며 missing_information에도 기록한다.
입력 meeting_id를 그대로 반환한다.""",

    ),
    ("human", "meeting_id: {meeting_id}\n회의록: {notes}"),
])

### 회의록 업무 정리 코드 작성하기

위 처리 흐름을 하나의 코드셀로 완성한다. 고객 문의 예제의 `Prompt → Model → schema → DataFrame` 구조를 재사용하되, 중첩된 `action_items`를 업무별 행으로 펼치고 담당자나 기한이 없는 업무를 별도로 찾아야 한다.


In [10]:
# 4) structured output 연결 + chain 구성
meeting_schema_model = model.with_structured_output(MeetingAnalysis)

# chain에 입력(dict) -> ChatPromptTemplate(Message)
# -> ChatModel(LLM) -> 출력(MeetingAnalysis)
meeting_chain = meeting_prompt | meeting_schema_model

# 모델 입력용 회의록 2건 준비
meeting_inputs = [
    {"meeting_id": c["meeting_id"], "notes": c["notes"]}
    for c in meeting_cases
]

# batch 입력은 각 입력이 독립적으로 API를 요청함
meeting_results = meeting_chain.batch(
    meeting_inputs,
    max_concurrency=2,  # 2개씩 병렬 처리
)

for result in meeting_results:
    print("meeting_id:", result.meeting_id)
    print("summary:", result.summary)
    print("decisions:", result.decisions)
    print("action_items:", result.action_items)
    print("missing_information:", result.missing_information)
    print("=" * 100)

meeting_id: M001
summary: 이번 배포에서 결제 페이지 오류를 수정하기로 했으며, 오류 화면 재설계와 결제 API 재시도 로직 수정 업무를 진행한다.
decisions: ['이번 배포에서 결제 페이지 오류를 수정한다.']
action_items: [MeetingAction(task='오류 화면을 다시 설계한다.', owner='민지', due_date='8월 12일'), MeetingAction(task='결제 API 재시도 로직을 수정한다.', owner='준호', due_date='8월 14일')]
missing_information: []
meeting_id: M002
summary: 추천 기능은 이번 배포에서 제외하고, 사용자 설문을 준비하며 API 비용을 검토하기로 했다.
decisions: ['추천 기능을 이번 배포에서 제외하기로 함']
action_items: [MeetingAction(task='사용자 설문 준비', owner='소연', due_date=None), MeetingAction(task='API 비용 검토', owner=None, due_date=None)]
missing_information: ['사용자 설문 마감일이 정해지지 않음', 'API 비용 검토 담당자가 정해지지 않음', 'API 비용 검토 기한이 정해지지 않음']


In [12]:
import pandas as pd
# 5) 중첩된 action_items를 업무별 행으로 펼쳐서 DataFrame 생성
rows = []
for result in meeting_results:
    for action in result.action_items:
        rows.append({
            "meeting_id": result.meeting_id,
            "task": action.task,
            "owner": action.owner,
            "due_date": action.due_date,
        })

action_frame = pd.DataFrame(rows)
print(action_frame.to_string(index=False))

meeting_id                 task owner due_date
      M001      오류 화면을 다시 설계한다.    민지   8월 12일
      M001 결제 API 재시도 로직을 수정한다.    준호   8월 14일
      M002            사용자 설문 준비    소연      NaN
      M002            API 비용 검토   NaN      NaN


In [13]:
# 6) None과 대표적인 누락 표현을 같은 값으로 정규화
missing_expressions = {None, "", "없음", "미정", "N/A", "미지정"}

def normalize_missing(value):
    return None if value in missing_expressions else value

action_frame["owner"] = action_frame["owner"].apply(normalize_missing)
action_frame["due_date"] = action_frame["due_date"].apply(normalize_missing)

# 정규화 후 owner나 due_date가 없는 업무를 Boolean 열로 표시
action_frame["owner_missing"] = action_frame["owner"].isna()
action_frame["due_date_missing"] = action_frame["due_date"].isna()

comparison_columns = ["meeting_id", "task", "owner", "owner_missing", "due_date", "due_date_missing"]
display(action_frame[comparison_columns])

# 추가 확인이 필요한 업무만 분리
needs_followup = action_frame[
    action_frame["owner_missing"] | action_frame["due_date_missing"]
]
print("추가 확인이 필요한 업무:")
display(needs_followup[comparison_columns])

print({
    "owner_missing_rate": action_frame["owner_missing"].mean(),
    "due_date_missing_rate": action_frame["due_date_missing"].mean(),
})

,meeting_id,task,owner,owner_missing,due_date,due_date_missing
0,M001,오류 화면을 다시 설계한다.,민지,False,8월 12일,False
1,M001,결제 API 재시도 로직을 수정한다.,준호,False,8월 14일,False
2,M002,사용자 설문 준비,소연,False,NaN,True
3,M002,API 비용 검토,NaN,True,NaN,True


추가 확인이 필요한 업무:


,meeting_id,task,owner,owner_missing,due_date,due_date_missing
2,M002,사용자 설문 준비,소연,False,NaN,True
3,M002,API 비용 검토,NaN,True,NaN,True


{'owner_missing_rate': np.float64(0.25), 'due_date_missing_rate': np.float64(0.5)}
